# 🤖 Telegram AI Project Builder Bot - Colab Runner

This notebook sets up and runs the Telegram AI Project Builder Bot in Google Colab.

## 📋 Instructions
1. Run all cells in order
2. Enter your Telegram Bot Token and OpenRouter API Key when prompted
3. The bot will start running in the background
4. Test by sending messages to your Telegram bot

## 🔑 New Feature: API Key Management
- Use `/apikey` command in Telegram to manage your API keys
- Add multiple keys, select which one to use, test them
- Keys are saved per-user and persist between sessions
- No need to re-enter keys each time

## ⚠️ Important Notes
- **Local Model**: Uses Qwen2.5 7B Instruct (as requested by user) - requires ~8GB VRAM
  - Falls back to Phi-2 automatically if insufficient GPU memory
  - Can override with `LOCAL_MODEL_NAME` environment variable
- **GPT-3.5 Turbo (OpenRouter)**: Fast and reliable via OpenRouter API (free tier available)
- **Auto Mode**: Tries multiple free models: openrouter/free, Gemini Pro, Llama 3.1, Phi-2
- You need an OpenRouter API key from https://openrouter.ai (free tier available)
- You need a Telegram Bot Token from @BotFather

---


In [ ]:
# @title 📦 Install Dependencies
!pip install -q python-telegram-bot python-dotenv flask flask-sqlalchemy flask-cors requests
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate bitsandbytes

print("✅ Dependencies installed!")

In [ ]:
# @title 📥 Clone Repository
repo_url = "https://github.com/guiseppeterry31-jpg/telegram-ai-project-builder.git"  # @param {type:"string"}

import os
!git clone $repo_url
os.chdir("telegram-ai-project-builder")

print("✅ Repository cloned!")

In [ ]:
# @title 🔑 Enter API Keys
import os

# Telegram Bot Token
telegram_token = input("Enter your Telegram Bot Token: ").strip()
os.environ['TELEGRAM_BOT_TOKEN'] = telegram_token

# OpenRouter API Key
openrouter_key = input("Enter your OpenRouter API Key: ").strip()
os.environ['OPENROUTER_API_KEY'] = openrouter_key

print("✅ API keys saved to environment!")

In [ ]:
# @title 🔧 Configure Environment
import os

# Create .env file
env_content = f"""TELEGRAM_BOT_TOKEN={telegram_token}
OPENROUTER_API_KEY={openrouter_key}
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ .env file created!")

In [ ]:
# @title 🧪 Test AI Models
import sys
sys.path.insert(0, '.')

print("Testing AI model configurations...")

# Check model configurations
print("\n📋 Current Model Configurations:")
print("-" * 40)

# Read model_router.py
with open('ai/model_router.py', 'r', encoding='utf-8') as f:
    content = f.read()
    if "openrouter/free" in content:
        print("[OK] Model router using 'openrouter/free'")
    else:
        print("[ERROR] Model router configuration issue")

# Read auto_rotate.py
with open('ai/auto_rotate.py', 'r', encoding='utf-8') as f:
    content = f.read()
    if "openrouter/free" in content:
        print("[OK] Auto rotate includes 'openrouter/free' as first model")
    else:
        print("[ERROR] Auto rotate configuration issue")

# Read local_mistral.py
with open('ai/local_mistral.py', 'r', encoding='utf-8') as f:
    content = f.read()
    if "microsoft/phi-2" in content:
        print("[OK] Local model using 'microsoft/phi-2' (2.7B parameters)")
    else:
        print("[ERROR] Local model configuration issue")

print("\n✅ Model configurations verified!")

In [ ]:
# @title 🚀 Start the Bot (Background)
import subprocess
import threading
import time

def run_bot():
    """Run the bot in a separate thread"""
    try:
        result = subprocess.run(
            ['python', 'main.py'],
            capture_output=True,
            text=True,
            timeout=None
        )
        print(result.stdout)
        if result.stderr:
            print("Errors:", result.stderr)
    except Exception as e:
        print(f"Bot error: {e}")

# Start bot in background thread
bot_thread = threading.Thread(target=run_bot, daemon=True)
bot_thread.start()

print("🤖 Bot is starting in the background...")
print("📱 Test it by sending /start to your bot on Telegram!")
print("⏳ The bot will keep running until this Colab runtime is disconnected.")
print("\n💡 Recommended Model Options:")
print("1. GPT-3.5 Turbo (OpenRouter) - Fast and reliable")
print("2. Auto Mode - Tries multiple free models")
print("3. Local Model (Colab) - Uses Phi-2 if GPU available")

# Keep the cell running
try:
    while bot_thread.is_alive():
        time.sleep(1)
except KeyboardInterrupt:
    print("Bot stopped.")

In [ ]:
# @title 📊 Check Bot Status
import requests
import os

telegram_token = os.environ.get('TELEGRAM_BOT_TOKEN', '')

if telegram_token:
    try:
        response = requests.get(f"https://api.telegram.org/bot{telegram_token}/getMe")
        if response.status_code == 200:
            bot_info = response.json()
            print(f"✅ Bot is active: @{bot_info['result']['username']}")
            print(f"   Name: {bot_info['result']['first_name']}")
        else:
            print(f"❌ Bot token may be invalid. Status: {response.status_code}")
    except Exception as e:
        print(f"❌ Error checking bot: {e}")
else:
    print("❌ No Telegram token found in environment.")

In [ ]:
# @title 📝 Test Instructions
print("📱 How to test the bot:")
print("1. Open Telegram and find your bot")
print("2. Send /start command")
print("3. Select a model option:")
print("   • Local Model (Colab) - Small local model")
print("   • GPT-3.5 Turbo (OpenRouter) - Fast via OpenRouter")
print("   • Auto Mode - Tries multiple free models")
print("4. Send a project request like: 'make a todo app' or 'create a website'")
print("5. Wait for the bot to generate and send a ZIP file")
print("\n🔑 API Key Management:")
print("   • Use /apikey command to manage your API keys:")
print("     - /apikey add <name> <key> - Add a new API key")
print("     - /apikey list - List your saved keys")
print("     - /apikey select <name> - Select which key to use")
print("     - /apikey test <name> - Test if a key works")
print("     - /apikey remove <name> - Remove a key")
print("     - /apikey status - Show current key status")
print("   • Keys are saved per-user and persist between sessions")
print("\n💡 Tip: For best results in Colab, use 'GPT-3.5 Turbo (OpenRouter)' or 'Auto Mode'")

In [ ]:
# @title 🔧 Troubleshooting
print("🔧 Common Issues & Solutions:")
print("\n1. Bot not responding:")
print("   • Check if the bot thread is still running")
print("   • Verify Telegram token is correct")
print("   • Make sure you sent /start to the bot")

print("\n2. AI models not working:")
print("   • Verify OpenRouter API key is valid")
print("   • Check if you have credits on OpenRouter (free tier available)")
print("   • Use 'Auto Mode' - it tries multiple models")

print("\n3. Local model failing in Colab:")
print("   • Colab may not have enough GPU memory for local models")
print("   • Switch to 'GPT-3.5 Turbo (OpenRouter)' or 'Auto Mode'")
print("   • The local model will automatically fallback to OpenRouter")

print("\n4. Project generation errors:")
print("   • Make sure your request is clear (e.g., 'make a todo app')")
print("   • The Auto-Enhancer will expand simple requests")
print("   • Check the bot output for specific error messages")

## 🎯 Features

- **Auto-Enhancer**: Turns simple requests into complete project specifications
- **Multi-Model Support**: Local Model (Phi-2), GPT-3.5 Turbo (OpenRouter), Auto Mode (tries 4 free models)
- **Project Generation**: Creates complete multi-file projects with proper structure
- **ZIP Export**: Sends generated projects as downloadable ZIP files via Telegram
- **Colab Ready**: Optimized for Google Colab with free model support

## 📁 Project Structure

```
telegram-ai-project-builder/
├── main.py                    # Entry point
├── MASTER_PROMPT.txt         # Combined Auto-Enhancer + Bot + Developer prompts
├── bot/                      # Telegram bot handlers
├── ai/                       # AI model implementations
│   ├── model_router.py       # Routes requests to selected model
│   ├── openrouter.py         # OpenRouter API client
│   ├── local_mistral.py      # Local Phi-2 model
│   └── auto_rotate.py        # Auto mode with model rotation
├── generator/                # Project generation
├── utils/                    # Utilities
└── requirements.txt          # Dependencies
```

## 🔗 Links

- **GitHub Repository**: https://github.com/guiseppeterry31-jpg/telegram-ai-project-builder
- **OpenRouter API**: https://openrouter.ai (get free API key)
- **Telegram BotFather**: https://t.me/botfather (create bot token)

---

**Note**: Keep this Colab tab open to keep the bot running. The bot will stop when the Colab runtime is disconnected.